# scRareBench | High-Level scDML | Dataset 0 — BMMC (GSE194122 curated)

> **Reproducibility**: Notebook sourced from the project execution archive, with outputs cleared and the scRareBench dependency pinned to the reviewed D3/D4 taxonomy commit `cfdb720f5488b9ceef9a225b35fc7558bcb0251c`. Method configuration, method seed and dataset preprocessing settings are preserved.

This Google Colab notebook runs **scDML** on scRareBench Dataset 0 (`gse194122`) through the generic **MethodSpec → benchmark_method** API.

The scDML implementation remains completely user-side. scRareBench loads the curated benchmark, orchestrates independent method seeds, validates latent/cell alignment, evaluates each run, and produces the multi-seed delivery.

### Dataset 0 / scDML contract

This notebook preserves the settings from the previously executed `scDML_modify` Dataset 0 workflow:

- Dataset selector: `0` (`gse194122`)
- Curated cells: 89,199
- Benchmark label: `celltype` — never supplied to scDML
- Integration batch: `BATCH`
- Initial clustering: Leiden, resolution `3.0`
- scDML preprocessing: 2000 HVGs, normalize total `1e4`, log1p, batch-wise scaling (`max_value=10`), PCA 50
- Integration: fully unsupervised, `expect_num_cluster=None`
- Merge candidates: all cluster counts from 2 through the observed number of initial Leiden clusters
- `K_in=5`, `K_bw=10`, cosine neighbor metrics, `rule2`
- 50 training epochs, hard triplets, Euclidean metric, margin `0.2`
- Output dimension `32`, hidden embedding layer `[256]`
- Method seeds: `42, 123, 2026`
- Benchmark seed fixed at `42`

The earlier executed Dataset 0 run found 59 initial Leiden clusters. This notebook recomputes the initial clustering for each method run rather than hard-coding 59.

> Use a **GPU + High-RAM Colab runtime**. scDML is old software and parts of its preprocessing/training pipeline materialize dense matrices. For a smoke test, temporarily use `METHOD_SEEDS = [42]`.

## 1. Install scRareBench and scDML

The modern Colab workflow previously used current scientific packages and installed scDML itself with `--no-deps`. This notebook follows the same pattern and uses Leiden, so the old `louvain` dependency is not needed.


In [ ]:
import subprocess
import sys

SCRAREBENCH_GITHUB = "git+https://github.com/amirhossein-alishahi/scRareBench_.git@cfdb720f5488b9ceef9a225b35fc7558bcb0251c"
SCDML_GITHUB = "git+https://github.com/eleozzr/scDML.git"

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps", SCRAREBENCH_GITHUB
])

METHOD_RUNTIME_DEPENDENCIES = (
    "tables>=3.6.1",
    "annoy>=1.17.0",
    "hnswlib>=0.5.2",
    "pytorch-metric-learning>=0.9.95",
    "numpy-groupies>=0.9.14",
    "numexpr>=2.7.1",
    "networkx>=2.5",
)

from scrarebench.runtime import print_install_report, setup_runtime

install_report = setup_runtime(
    extra_requirements=METHOD_RUNTIME_DEPENDENCIES,
    extra_imports=(
        "tables", "annoy", "hnswlib", "pytorch_metric_learning",
        "numpy_groupies", "numexpr", "networkx",
    ),
    quiet=False,
)
print_install_report(install_report)

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps", SCDML_GITHUB
])

import scDML
from scDML import scDMLModel
print("scDML:", getattr(scDML, "__version__", "installed"))


In [ ]:
# OPTIONAL runtime diagnostics only.
#
# import sys
# import scanpy as sc
# import torch
# from importlib import metadata
#
# print("Python:", sys.version)
# print("scanpy:", sc.__version__)
# print("torch:", torch.__version__)
# print("CUDA:", torch.cuda.is_available())
# for pkg in ("numpy", "anndata", "pytorch-metric-learning", "hnswlib", "annoy"):
#     try:
#         print(pkg, metadata.version(pkg))
#     except metadata.PackageNotFoundError:
#         print(pkg, "not installed")


## 2. Load curated Dataset 0


In [ ]:
from scrarebench import load_dataset, dataset_info

adata = load_dataset(0)
info = dataset_info(adata)

print(adata)
print("\nDataset metadata:")
for key, value in info.items():
    print(f"  {key}: {value}")

assert adata.n_obs == 89_199, f"Unexpected Dataset 0 cell count: {adata.n_obs}"
assert info["label_key"] == "celltype"
assert info["batch_key"] == "BATCH"
assert adata.obs_names.is_unique


## 3. Reproducibility and configuration

The method seed changes between runs, while the benchmark seed remains fixed. The preprocessing and integration parameters below match the previously executed Dataset 0 scDML workflow.


In [ ]:
METHOD_SEEDS = [42, 123, 2026]
BENCHMARK_SEED = 42

METHOD_CONFIG = {
    "cluster_method": "leiden",
    "resolution": 3.0,
    "n_high_var": 2000,
    "normalize_samples": True,
    "target_sum": 1e4,
    "log_normalize": True,
    "normalize_features": True,
    "pca_dim": 50,
    "scale_value": 10.0,
    "K_in": 5,
    "K_bw": 10,
    "K_in_metric": "cosine",
    "K_bw_metric": "cosine",
    "merge_rule": "rule2",
    "num_epochs": 50,
    "projection": False,
    "early_stop": False,
    "batch_size": 64,
    "metric": "euclidean",
    "margin": 0.2,
    "triplet_type": "hard",
    "device": None,
    "out_dim": 32,
    "emb_dim": [256],
    "save_model": False,
}

OUTPUT_DIR = "/content/scrarebench_scdml_dataset0_highlevel"
METHOD_WORK_DIR = "/content/scrarebench_scdml_method_runs"


## 4. User-owned scDML runner

scDML's own preprocessing performs batch-wise concatenation, which can change row order and observation names. This runner stores the original Dataset 0 barcode before preprocessing and explicitly reorders the final `X_emb` representation back to the original benchmark order before returning it to scRareBench.


In [ ]:
from pathlib import Path
import gc
import random

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch

from scDML import scDMLModel
from scrarebench import MethodOutput

_ORIGINAL_BARCODE_COL = "_scrarebench_original_barcode"

def _looks_like_raw_counts(matrix, max_values=100_000, seed=42):
    values = matrix.data if sp.issparse(matrix) else np.asarray(matrix).ravel()
    values = np.asarray(values)
    if len(values) > max_values:
        rng = np.random.default_rng(int(seed))
        values = values[rng.choice(len(values), size=max_values, replace=False)]
    values = values.astype(float, copy=False)
    if len(values) == 0:
        return True
    if not np.isfinite(values).all():
        return False
    return bool(values.min() >= 0 and np.allclose(values, np.rint(values), atol=1e-6))

def run_scdml(method_adata, seed, config):
    batch_key = info["batch_key"]
    count_layer = info.get("count_layer") or "counts"

    original_order = method_adata.obs_names.astype(str).to_numpy(copy=True)
    if len(np.unique(original_order)) != len(original_order):
        raise ValueError("Dataset 0 observation names must be unique.")

    method_adata.obs[_ORIGINAL_BARCODE_COL] = original_order

    if "feature_types" in method_adata.var.columns:
        ft = method_adata.var["feature_types"].astype(str).str.upper()
        rna_mask = ft.eq("GEX").to_numpy()
        if rna_mask.sum() == 0:
            raise ValueError("No GEX features were found.")
        if rna_mask.sum() != method_adata.n_vars:
            method_adata = method_adata[:, rna_mask].copy()

    method_adata.var_names_make_unique()

    if count_layer not in method_adata.layers:
        raise KeyError(
            f"Raw-count layer {count_layer!r} is required for scDML. "
            f"Available layers: {list(method_adata.layers)}"
        )

    counts = method_adata.layers[count_layer]
    if not _looks_like_raw_counts(counts, seed=seed):
        raise ValueError(f"adata.layers[{count_layer!r}] does not look like raw counts.")

    # scDML.preprocess() consumes adata.X.
    method_adata.X = counts.copy()

    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))

    seed_dir = Path(METHOD_WORK_DIR) / f"seed_{int(seed)}"
    seed_dir.mkdir(parents=True, exist_ok=True)

    model = scDMLModel(verbose=True, save_dir=str(seed_dir) + "/")

    adata_scdml = model.preprocess(
        method_adata,
        cluster_method=str(config["cluster_method"]),
        resolution=float(config["resolution"]),
        batch_key=batch_key,
        n_high_var=int(config["n_high_var"]),
        normalize_samples=bool(config["normalize_samples"]),
        target_sum=float(config["target_sum"]),
        log_normalize=bool(config["log_normalize"]),
        normalize_features=bool(config["normalize_features"]),
        pca_dim=int(config["pca_dim"]),
        scale_value=float(config["scale_value"]),
        mode="unsupervised",
    )

    if _ORIGINAL_BARCODE_COL not in adata_scdml.obs:
        raise RuntimeError("The original-barcode guard column was lost during scDML preprocessing.")

    num_init_clusters = int(adata_scdml.obs["init_cluster"].nunique())
    if num_init_clusters < 2:
        raise RuntimeError(f"Too few initial clusters: {num_init_clusters}")

    ncluster_candidates = list(range(2, num_init_clusters + 1))
    print(
        f"Seed {seed}: initial Leiden clusters={num_init_clusters}; "
        f"merge candidates=2..{num_init_clusters}"
    )

    model.integrate(
        adata_scdml,
        batch_key=batch_key,
        ncluster_list=ncluster_candidates,
        expect_num_cluster=None,
        K_in=int(config["K_in"]),
        K_bw=int(config["K_bw"]),
        K_in_metric=str(config["K_in_metric"]),
        K_bw_metric=str(config["K_bw_metric"]),
        merge_rule=str(config["merge_rule"]),
        num_epochs=int(config["num_epochs"]),
        projection=bool(config["projection"]),
        early_stop=bool(config["early_stop"]),
        batch_size=int(config["batch_size"]),
        metric=str(config["metric"]),
        margin=float(config["margin"]),
        triplet_type=str(config["triplet_type"]),
        device=config["device"],
        seed=int(seed),
        out_dim=int(config["out_dim"]),
        emb_dim=list(config["emb_dim"]),
        save_model=bool(config["save_model"]),
        celltype_key=None,
        mode="unsupervised",
    )

    if "X_emb" not in adata_scdml.obsm:
        raise RuntimeError("scDML did not create adata.obsm['X_emb'].")

    latent_native = np.asarray(adata_scdml.obsm["X_emb"], dtype=np.float32)
    native_barcodes = adata_scdml.obs[_ORIGINAL_BARCODE_COL].astype(str).to_numpy()

    if latent_native.ndim != 2 or latent_native.shape[0] != len(native_barcodes):
        raise RuntimeError(f"Unexpected scDML latent shape: {latent_native.shape}")
    if not np.isfinite(latent_native).all():
        raise ValueError("scDML latent contains NaN or infinite values.")
    if len(np.unique(native_barcodes)) != len(native_barcodes):
        raise RuntimeError("Original barcodes are no longer unique after preprocessing.")
    if set(native_barcodes) != set(original_order):
        raise RuntimeError("scDML preprocessing changed the benchmark cell set.")

    position = pd.Series(np.arange(len(native_barcodes)), index=native_barcodes)
    reorder = position.loc[original_order].to_numpy()
    latent = latent_native[reorder]

    if not np.array_equal(native_barcodes[reorder], original_order):
        raise RuntimeError("Failed to restore scDML latent to original Dataset 0 order.")

    provenance = {}
    log_path = seed_dir / "log.txt"
    if log_path.exists():
        provenance["scdml_log"] = log_path

    print(f"Seed {seed}: aligned latent shape={latent.shape}")

    del model, adata_scdml, latent_native, position, reorder
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return MethodOutput(
        latent=latent,
        barcodes=original_order,
        representation_key="X_scdml",
        provenance_files=provenance,
    )


## 5. Run the three-seed High-Level benchmark

For an initial compatibility test, temporarily change `METHOD_SEEDS` to `[42]`. Restore all three seeds for the final benchmark.


In [ ]:
from scrarebench import MethodSpec, benchmark_method

method = MethodSpec(
    name="scDML",
    runner=run_scdml,
    config=METHOD_CONFIG,
    dependencies=METHOD_RUNTIME_DEPENDENCIES + (SCDML_GITHUB,),
)

result = benchmark_method(
    adata,
    method,
    seeds=METHOD_SEEDS,
    output_dir=OUTPUT_DIR,
    benchmark_config={
        "random_state": BENCHMARK_SEED,
        "run_scib": True,
        "scib_n_hvg": 4000,
        "scib_reference_n_pcs": 50,
        "scib_n_jobs": 1,
        "scib_progress_bar": True,
        "scib_include_silhouette_batch": True,
        "scib_require_success": True,
        "include_latent_in_bundle": False,
    },
    install_dependencies=False,
    finalize=True,
)


## 6. Inspect the multi-seed summary


In [ ]:
from IPython.display import display

display(result.summary().round(5))
print("Method:", result.method)
print("Completed seeds:", result.seeds)
print("Multi-seed report:", result.report_path)
print("Delivery archive:", result.archive_path)
print("Summary JSON:", result.summary_path)

for seed, run in result.runs.items():
    print(f"\nSeed {seed}")
    print("  representation:", run.representation_key)
    print("  output:", run.output_dir)
    print("  interactive report:", run.interactive_report_path)
    print("  PDF:", run.pdf_path)
    print("  bundle:", run.bundle_path)


## 7. Download the final multi-seed archive


In [ ]:
from pathlib import Path
from google.colab import files

archive_path = Path(result.archive_path)
if not archive_path.exists():
    raise FileNotFoundError(
        f"Multi-seed output archive was not found: {archive_path}\n"
        "Make sure benchmark_method(..., finalize=True) completed successfully."
    )

print(f"Downloading: {archive_path.name}")
print(f"Size: {archive_path.stat().st_size / (1024**2):.2f} MB")
files.download(str(archive_path))
